# The page-16 avoided-emissions figure, under a grid of methods

ATAG *Waypoint 2050* (third edition), page 16, reports that "efficiency measures
have already saved 14.6 Gt of CO2 since 1990". This notebook reproduces that
number, establishes the four undeclared choices behind it, and then relaxes the
one assumption the construction cannot survive: that traffic would have been
the same in a world where flying cost more than twice as much.

Everything is read from committed files. No model runs here. Outputs land in
`data_outputs/` and are what the document reads.

In [ ]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

HERE = Path.cwd()
sys.path.insert(0, str(HERE / "models"))
import frozen_baseline as fb  # noqa: E402

OUTPUTS = HERE / "data_outputs"
OUTPUTS.mkdir(exist_ok=True)

with open(HERE.parent / "params_elasticity.yaml") as handle:
    PARAMS = yaml.safe_load(handle)

EPSILON = PARAMS["central"]
PASS_THROUGH = PARAMS["pass_through"]
FUEL_SHARE = PARAMS["fuel_cost_share"]["central"]
LF_RECOVERY = PARAMS["load_factor_recovery"]

print(
    f"elasticity {EPSILON}, pass-through {PASS_THROUGH}, "
    f"fuel share of fare {FUEL_SHARE}, load-factor recovery {LF_RECOVERY}"
)

## The R0 gate

The reproduction has to hit all four quantities the figure labels before any
counterfactual built on it is worth reading. Tolerance is 3 %, which is wider
than the reproduction error and narrower than the differences between methods
that the rest of the notebook is about.

The four are not independent: one scope factor, 0.8320, has to reconcile the
observed 2019 value, the 2050 frozen endpoint and the 2050 no-effort value
simultaneously. That is what identifies the published series as tank-to-wake
rather than well-to-wake.

In [ ]:
observed, splice = fb.observed_co2()
r0 = fb.frozen_baseline()

no_effort_2050 = float(fb.scenario_co2("t0-TTW").loc[2050])
frozen_2050_endpoint = fb.frozen_baseline(anchor="endpoint")["counterfactual_at_horizon"]

gate = pd.DataFrame(
    [
        ("Frozen 1990 efficiency at 2050", fb.ATAG_FROZEN_2050_MT, frozen_2050_endpoint, "Mt"),
        ("Observed CO2 in 2019", fb.ATAG_OBSERVED_2019_MT, float(observed.loc[2019]), "Mt"),
        ("2050 without additional efforts", fb.ATAG_NO_EFFORT_2050_MT, no_effort_2050, "Mt"),
        ("Avoided since 1990", fb.ATAG_AVOIDED_GT, r0["avoided_gt"], "Gt"),
    ],
    columns=["quantity", "reported", "reproduced", "unit"],
)
gate["error_pct"] = 100.0 * (gate["reproduced"] / gate["reported"] - 1.0)
display(gate.round(2))

assert (gate["error_pct"].abs() < 3.0).all(), gate
print("R0 gate passed: all four labelled quantities within 3 %")

### What the gate already settles

Two readings of the frozen line are compatible with the figure's own labels, and
only one of them survives the 2050 endpoint. If only the fuel-side intensities
were held at 1990 and load factor were allowed to improve as it actually did,
the frozen line would be scaled by ASK rather than RPK and would land far below
the labelled ~5,200 Mt at any plausible 1990 anchor.

So load factor is inside the retrospective "efficiency" credit. The report's own
text agrees: it places "higher load factors and optimised aircraft cabin usage"
inside the 25 % operational efficiency gain. It is then credited a second time as
a forward mitigation lever, and the size of that double count is the difference
between the two rows below.

In [ ]:
freeze_all = fb.frozen_baseline(frozen_factors=fb.METHODS["freeze-all"])
fuel_side = fb.frozen_baseline(frozen_factors=fb.METHODS["fuel-side"])

double_count = freeze_all["avoided_gt"] - fuel_side["avoided_gt"]
print(
    f"freeze all three factors (RPK-scaled):  {freeze_all['avoided_gt']:6.2f} Gt, "
    f"2050 endpoint {freeze_all['counterfactual_at_horizon']:.0f} Mt"
)
print(
    f"fuel-side only    (ASK-scaled):         {fuel_side['avoided_gt']:6.2f} Gt, "
    f"2050 endpoint {fuel_side['counterfactual_at_horizon']:.0f} Mt"
)
print(
    f"load factor counted twice:              {double_count:6.2f} Gt "
    f"({100 * double_count / freeze_all['avoided_gt']:.0f} % of the headline)"
)

assert fuel_side["counterfactual_at_horizon"] < 0.85 * fb.ATAG_FROZEN_2050_MT, (
    "the ASK-scaled reading should be ruled out by the figure's own 2050 label"
)

### The report's own arithmetic

Page 16 states ~29 % efficiency gain from aircraft technology and 25 % from
operations since 1990, then reports 54 % combined. Two efficiency factors
compose multiplicatively. Adding them instead is the same non-additivity that
the sequential lever attribution suffers from, showing up in the retrospective
headline.

In [ ]:
composition = fb.stated_gain_composition()
for key, value in composition.items():
    print(f"{key:>40}: {value}")

assert composition["reading"] == "additive", (
    "the stated 54 % is no longer the arithmetic sum of 29 % and 25 %; "
    "recheck the report text before the document repeats this finding"
)
print()
print(
    f"stated 54 % implies a frozen-to-observed ratio of "
    f"{composition['implied_frozen_ratio_stated']:.2f}"
)
print(
    f"composing the two factors properly implies "
    f"{composition['implied_frozen_ratio_multiplicative']:.2f}"
)
print(f"observed ratio in {r0['years'][-1]}: {r0['counterfactual'][-1] / r0['observed'][-1]:.2f}")

## The method grid

Four choices, none of them stated in the report, each of them material:

1. **which factors are frozen**, which selects the observed activity series the
   1990 intensity is scaled by;
2. **the window** the shaded area is integrated over;
3. **the 1990 anchor**, where the data-derived and figure-derived values differ
   by 3.5 %;
4. **where the observed series is spliced**, since no single source covers 1990
   to 2024.

The grid is the honest object. The published figure reports one cell of it.

In [ ]:
rows = []
for method, factors in fb.METHODS.items():
    for end in (2019, 2023, 2024):
        for anchor in ("klower", "endpoint"):
            for splice_year, rescale in ((2019, False), (2000, False), (2019, True)):
                cell = fb.frozen_baseline(
                    frozen_factors=factors,
                    window=(1990, end),
                    anchor=anchor,
                    splice_year=splice_year,
                    rescale_klower=rescale,
                )
                rows.append(
                    {
                        "method": method,
                        "driver": cell["config"]["driver"],
                        "window_end": end,
                        "anchor": anchor,
                        "e_1990": round(cell["config"]["e_1990"], 2),
                        "splice_year": splice_year,
                        "rescale_klower": rescale,
                        "avoided_gt": cell["avoided_gt"],
                        "frozen_2050_mt": cell["counterfactual_at_horizon"],
                    }
                )

grid = pd.DataFrame(rows)
grid.to_csv(OUTPUTS / "grid.csv", index=False)

published = grid.query(
    "method == 'freeze-all' and window_end == 2023 and anchor == 'klower' "
    "and splice_year == 2019 and not rescale_klower"
)["avoided_gt"].item()
print(
    f"{len(grid)} cells; avoided ranges "
    f"{grid['avoided_gt'].min():.2f} to {grid['avoided_gt'].max():.2f} Gt"
)
print(f"the cell that reproduces the published figure: {published:.2f} Gt against 14.6 Gt reported")

headline = grid.query("anchor == 'klower' and splice_year == 2019 and not rescale_klower")
display(headline.pivot(index="window_end", columns="method", values="avoided_gt").round(2))

### The same grid, as curves

The document draws a band rather than a line, so it needs the grid per year and
not only its integral. Every cell is run out to 2024, the last observed year, so
one file carries the observed series and every counterfactual on the same index.
The 2050 endpoints go alongside, since the frozen line's own label lives there.

In [ ]:
FULL = (1990, 2024)
variants = {}
for method, factors in fb.METHODS.items():
    for anchor in ("klower", "endpoint"):
        for splice_year, rescale in ((2019, False), (2000, False), (2019, True)):
            tag = f"{method}|{anchor}|splice{splice_year}{'|rescaled' if rescale else ''}"
            variants[tag] = fb.frozen_baseline(
                frozen_factors=factors,
                window=FULL,
                anchor=anchor,
                splice_year=splice_year,
                rescale_klower=rescale,
            )

reference = variants["freeze-all|klower|splice2019"]
series = pd.DataFrame(
    {"observed": reference["observed"]},
    index=pd.Index(reference["years"], name="year"),
)
for tag, cell in variants.items():
    series[tag] = cell["counterfactual"]

for name, run in (
    ("R0", fb.frozen_baseline(window=FULL)),
    (
        "R1",
        fb.frozen_baseline(
            window=FULL,
            adaptation="demand",
            elasticity=EPSILON,
            pass_through=PASS_THROUGH,
            fuel_cost_share=FUEL_SHARE,
        ),
    ),
    (
        "R2",
        fb.frozen_baseline(
            window=FULL,
            adaptation="supply",
            elasticity=EPSILON,
            pass_through=PASS_THROUGH,
            fuel_cost_share=FUEL_SHARE,
            lf_adjustment=LF_RECOVERY,
        ),
    ),
):
    series[name] = run["counterfactual"]
    series[f"{name}_traffic_ratio"] = run["demand_ratio"]

series.to_csv(OUTPUTS / "series.csv")

envelope = series[list(variants)]
print(
    f"{len(variants)} curves; 2023 counterfactual spans "
    f"{envelope.loc[2023].min():.0f} to {envelope.loc[2023].max():.0f} Mt "
    f"against {series.loc[2023, 'observed']:.0f} Mt observed"
)

horizons = pd.DataFrame(
    [
        {"variant": tag, "frozen_2050_mt": cell["counterfactual_at_horizon"]}
        for tag, cell in variants.items()
    ]
).dropna()
for name, run in (
    ("R0", fb.frozen_baseline(window=FULL)),
    (
        "R1",
        fb.frozen_baseline(
            window=FULL,
            adaptation="demand",
            elasticity=EPSILON,
            pass_through=PASS_THROUGH,
            fuel_cost_share=FUEL_SHARE,
        ),
    ),
    (
        "R2",
        fb.frozen_baseline(
            window=FULL,
            adaptation="supply",
            elasticity=EPSILON,
            pass_through=PASS_THROUGH,
            fuel_cost_share=FUEL_SHARE,
            lf_adjustment=LF_RECOVERY,
        ),
    ),
):
    horizons.loc[len(horizons)] = [name, run["counterfactual_at_horizon"]]
horizons.to_csv(OUTPUTS / "horizons.csv", index=False)
print(
    f"2050 endpoints span {horizons['frozen_2050_mt'].min():.0f} to "
    f"{horizons['frozen_2050_mt'].max():.0f} Mt against the labelled "
    f"{fb.ATAG_FROZEN_2050_MT:.0f} Mt"
)

### The two forward legs the same figure draws

Page 16 also carries the report's own prospective legs, and the redrawn figure has to show them for
the comparison to be like for like: the trajectory reaching net zero in 2050, and the dashed
"2050 emissions without additional efforts" ending near 2,400 Mt. Both come straight out of the
companion reproduction's committed scenarios, `s1-TTW` and `t0-TTW`, and are read here rather than
recomputed.

In [ ]:
forward = pd.DataFrame(
    {
        "net_zero": fb.scenario_co2("s1-TTW"),
        "no_effort": fb.scenario_co2("t0-TTW"),
    }
).loc[2023:2050]
forward.to_csv(OUTPUTS / "forward.csv")
print(forward.loc[[2023, 2030, 2040, 2050]].round(1))

### The seam

The two observed sources disagree by a near-constant factor over the nine years
they overlap. Splicing at 2019 uses Kloewer wherever it exists, which is the
reading that reproduces 14.6 Gt, and leaves a step at the seam. Removing the
step by putting Kloewer on the reproduction's coverage basis moves the 1990
anchor and therefore the headline. Both are reported; neither is picked.

In [ ]:
for key, value in splice.items():
    print(f"{key:>24}: {value}")
print()
print(
    f"seam step at {splice['splice_year'] - 1}/{splice['splice_year']}: "
    f"{100 * splice['seam_discontinuity']:.1f} %"
)

for label, kwargs in (
    ("default splice 2019, no rescale", {}),
    ("splice 2000, no rescale", {"splice_year": 2000}),
    ("splice 2019, Kloewer rescaled", {"rescale_klower": True}),
):
    cell = fb.frozen_baseline(**kwargs)
    print(
        f"{label:>34}: E1990 {cell['config']['e_1990']:6.1f} Mt, "
        f"avoided {cell['avoided_gt']:5.2f} Gt"
    )

## R0, R1, R2

R0 is the published construction. R1 relaxes exactly one assumption, that
traffic does not respond to price, and holds fleet and technology frozen exactly
as R0 does; **R0 minus R1 is therefore the direct rebound in isolation**. R2 adds
the cheapest supply-side response available under frozen 1990 technology, which
is filling the aircraft the airline already has.

The ordering `R2 < R1 < R0` is asserted, not hoped for: if it failed, the
counterfactual bracket would not be a bracket.

In [ ]:
runs = {
    "r0": fb.frozen_baseline(),
    "r1": fb.frozen_baseline(
        adaptation="demand",
        elasticity=EPSILON,
        pass_through=PASS_THROUGH,
        fuel_cost_share=FUEL_SHARE,
    ),
    "r2": fb.frozen_baseline(
        adaptation="supply",
        elasticity=EPSILON,
        pass_through=PASS_THROUGH,
        fuel_cost_share=FUEL_SHARE,
        lf_adjustment=LF_RECOVERY,
    ),
}

summary = pd.DataFrame(
    [
        {
            "run": name,
            "adaptation": run["config"]["adaptation"],
            "avoided_gt": run["avoided_gt"],
            "counterfactual_2023_mt": run["counterfactual"][-1],
            "traffic_vs_observed_2023": run["demand_ratio"][-1],
            "frozen_2050_mt": run["counterfactual_at_horizon"],
        }
        for name, run in runs.items()
    ]
)
display(summary.round(3))

assert runs["r1"]["avoided_gt"] < runs["r0"]["avoided_gt"], "R1 must sit below R0"
assert runs["r2"]["avoided_gt"] < runs["r1"]["avoided_gt"], "R2 must sit below R1"

overstatement = fb.ATAG_AVOIDED_GT / runs["r1"]["avoided_gt"] - 1.0
print(f"observed CO2 in 2023: {runs['r0']['observed'][-1]:.0f} Mt")
print(
    f"R0 counterfactual:    {runs['r0']['counterfactual'][-1]:.0f} Mt "
    f"(factor {runs['r0']['counterfactual'][-1] / runs['r0']['observed'][-1]:.2f})"
)
print(f"the published 14.6 Gt is {100 * overstatement:.0f} % above R1 at eps = {EPSILON}")

## Epsilon star

Rather than argue about which elasticity is right, ask which elasticity the
published figure requires. If the answer lies outside every published estimate,
the figure is indefensible under any plausible parameterisation, and the burden
moves: a reviewer has to defend a number, not merely doubt ours.

In [ ]:
eps_star = fb.epsilon_star(
    pass_through=PASS_THROUGH,
    fuel_cost_share=FUEL_SHARE,
)
print(f"epsilon* reproducing 14.6 Gt: {eps_star:.4f}")

share_sensitivity = pd.DataFrame(
    [
        {
            "fuel_cost_share": share,
            "epsilon_star": fb.epsilon_star(pass_through=PASS_THROUGH, fuel_cost_share=share),
        }
        for share in PARAMS["fuel_cost_share"]["sweep"]
    ]
)
display(share_sensitivity.round(4))

literature = {key: value["range"] for key, value in PARAMS["sources"].items()}
worst_case = max(hi for _, hi in literature.values())
print()
for key, (lo, hi) in literature.items():
    print(f"{key:>22}: {lo} to {hi}")
print(f"\nleast elastic estimate in the literature: {worst_case}")
print(f"epsilon* is {abs(worst_case / eps_star):.0f} times smaller in magnitude")

assert eps_star > worst_case, (
    "epsilon* fell inside the published range; the argument in the document "
    "would then need to be stated as a point estimate rather than a bound"
)

In [ ]:
sweep = np.linspace(PARAMS["sweep"]["low"], PARAMS["sweep"]["high"], PARAMS["sweep"]["steps"])
curve = pd.DataFrame(
    [
        {
            "elasticity": float(eps),
            "avoided_gt": fb.frozen_baseline(
                adaptation="demand",
                elasticity=float(eps),
                pass_through=PASS_THROUGH,
                fuel_cost_share=FUEL_SHARE,
            )["avoided_gt"],
        }
        for eps in sweep
    ]
)
curve.to_csv(OUTPUTS / "elasticity_curve.csv", index=False)
print(
    f"swept {len(curve)} elasticities from {curve['elasticity'].min():.2f} "
    f"to {curve['elasticity'].max():.2f}"
)
display(curve.iloc[::10].round(3))

## Write the outputs

The document reads these three files and nothing else. Each carries the full
configuration that produced it, so a figure in the document cannot silently
drift from the run behind it.

In [ ]:
def to_json(run, extras=None):
    payload = {
        "config": run["config"],
        "years": [int(year) for year in run["years"]],
        "counterfactual_mt": [float(value) for value in run["counterfactual"]],
        "observed_mt": [float(value) for value in run["observed"]],
        "avoided_mt": [float(value) for value in run["avoided"]],
        "demand_ratio": [float(value) for value in run["demand_ratio"]],
        "avoided_gt": run["avoided_gt"],
        "counterfactual_at_horizon_mt": run["counterfactual_at_horizon"],
    }
    payload.update(extras or {})
    return payload


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:16]


inputs_digest = {
    str(path.relative_to(fb.REPO)).replace("\\", "/"): sha256(path)
    for path in (
        fb.TRAFFIC,
        fb.KLOWER,
        fb.FUEL_PRICE,
        fb.ATAG / "3rd_edition_full" / "data_outputs" / "s1-TTW.json",
        fb.ATAG / "3rd_edition_full" / "data_outputs" / "t0-TTW.json",
    )
}

extras = {
    "r0": {
        "gate": gate.to_dict(orient="records"),
        "load_factor_double_count_gt": double_count,
        "stated_gain_composition": composition,
        "grid_span_gt": [float(grid["avoided_gt"].min()), float(grid["avoided_gt"].max())],
    },
    "r1": {
        "epsilon_star": eps_star,
        "epsilon_star_by_fuel_share": share_sensitivity.to_dict(orient="records"),
        "literature_ranges": literature,
    },
    "r2": {},
}

for name, run in runs.items():
    payload = to_json(run, extras[name])
    payload["inputs_sha256"] = inputs_digest
    with open(OUTPUTS / f"{name}.json", "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)
    print(f"wrote data_outputs/{name}.json ({payload['avoided_gt']:.2f} Gt)")

## Nothing inherited was touched

This analysis reads Part A's committed outputs and writes only inside
`avoided_emissions/`. The check is mechanical rather than a promise: git is
asked whether anything outside this directory moved.

In [ ]:
try:
    changed = subprocess.run(
        ["git", "status", "--porcelain", "aeromaps/notebooks/scenarios/02_atag_waypoint2050"],
        cwd=fb.REPO,
        capture_output=True,
        text=True,
        check=True,
    ).stdout.splitlines()
except (OSError, subprocess.CalledProcessError) as error:
    print(f"git unavailable, skipping the untouched check: {error}")
else:
    stray = [line for line in changed if "avoided_emissions/" not in line.replace("\\", "/")]
    for line in changed:
        print(line)
    assert not stray, f"something outside avoided_emissions/ moved: {stray}"
    print("\nnothing outside avoided_emissions/ changed")